In [2]:
"""Phase 3 prep: split payments.csv into one file per day (like a real daily report drop)
and add realistic mess for Power Query to clean:
  - every 6th file has txn_type in UPPERCASE and status with trailing spaces
  - 30 rows are re-sent in the next day's file (duplicates)
"""
import os
import pandas as pd

# 1. Define your exact folder path so Python doesn't get lost
base_path = r"C:\Users\REDTECH\Desktop\fraud_hub\data"

# 2. Update the script to use that base_path
os.makedirs(f"{base_path}\daily_drops", exist_ok=True)
pay = pd.read_csv(f"{base_path}\payments.csv")

pay["day"] = pay["txn_time"].str[:10]
days = sorted(pay["day"].unique())
resend = pay.sample(30, random_state=7)          # rows that get sent twice

total_rows = 0
for i, d in enumerate(days):
    chunk = pay[pay["day"] == d].drop(columns="day").copy()
    if i > 0:
        prev = resend[resend["day"] == days[i - 1]].drop(columns="day")
        chunk = pd.concat([chunk, prev])          # duplicates arrive next day
    if i % 6 == 5:
        chunk["txn_type"] = chunk["txn_type"].str.upper()
        chunk["status"] = chunk["status"] + "  "
    
    # 3. Save the new files into the correct folder
    chunk.to_csv(f"{base_path}\daily_drops\payments_{d}.csv", index=False)
    total_rows += len(chunk)

print(f"{len(days)} files written, {total_rows} rows total "
      f"({len(pay)} unique + {total_rows - len(pay)} duplicates)")

<>:13: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:14: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:31: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:13: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:14: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:31: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\REDTECH\AppData\Local\Temp\ipykernel_24688\4251040072.py:13: Syntax

183 files written, 50682 rows total (50652 unique + 30 duplicates)
